### FFT Example from SciPy:
https://docs.scipy.org/doc/scipy/reference/tutorial/fft.html

In [ ]:
#import packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
#import scipy.fftpack
#import scipy.fft
import os
import csv
import re
import glob
import matplotlib.patches as patches
from fourier_functions import *
from coordinate_conversion_functions import *
from track_plotting_daniel import *

In [ ]:
#this is the intial funciton constructed out of 2 sinoid functions y1 and y2 (see below)
N = 200 #number of samplepoints
# sample spacing
T = 1.0 / 800.0
x = np.linspace(0.0, N*T, N) #array from 0 to 0.25 with 200 values
y = np.sin(50.0 * 2*np.pi*x) + 0.5*np.sin(120.0 * 2.0*np.pi*x)

plt.figure(figsize=(7,2), dpi=150)
plt.plot(y)
plt.ylabel('Amplitude')
plt.xlabel('Time')
plt.show()

In [ ]:
#composite and consitutive functions
y1= np.sin(50.0 * 2.0*np.pi*x)
y2= 0.5*np.sin(120.0 * 2.0*np.pi*x)
plt.figure(figsize=(7,2), dpi=150)
plt.plot(y)
plt.plot(y1, alpha=0.5)
plt.plot(y2, alpha=0.5)
plt.ylabel('Amplitude')
plt.xlabel('Time')
plt.show()

In [ ]:
#here the fft does its magic
yf = scipy.fftpack.fft(y)
xf = scipy.fftpack.fftfreq(N, T)[:N//2]
fig, ax = plt.subplots(figsize=(7,2), dpi=150)
ax.plot(xf, 2.0/N * np.abs(yf[:N//2]))
plt.ylabel('Amplitude')
plt.xlabel('Frequency')
plt.show()

# fourier for parts of tracks of single worms

In [ ]:
#these are csvs i created with the head_position script 
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/head_tail_center_absolute_coordinates_x_ref_corrected/2020-07-01_14-41-11_chemotaxisl_worm2-TablePosRecord.csv'
df_head_tail_center_coords=pd.read_csv(path)

In [ ]:
#plot the track to determine on which parts in want to do the fourier transformations
figsize_x=10
figsize_y=5
line_width=0.5
plot_tracks_center_head_tail(path,figsize_x,figsize_y,line_width)

In [ ]:
#here i pull out timeframes where the worm is either awy form food chemotaxing or in the food, in order to compare 
#it with fourier transformation
away_from_food=df_head_tail_center_coords[(df_head_tail_center_coords['x_head_corrected'].between(11, 14)) & 
(df_head_tail_center_coords['y_head_corrected']< 18)]               
xmin=away_from_food['x_head_corrected'].min()
xmax=away_from_food['x_head_corrected'].max()
ymin=away_from_food['y_head_corrected'].min()
ymax=away_from_food['y_head_corrected'].max()

In [ ]:
print(away_from_food.index.min())
print(away_from_food.index.max())

In [ ]:
#plot the track to determine on which parts in want to do the fourier transformations
figsize_x=10
figsize_y=5
line_width=0.5
x_head_corrected=df_head_tail_center_coords['x_head_corrected']
y_head_corrected=df_head_tail_center_coords['y_head_corrected']
x_tail_corrected=df_head_tail_center_coords['x_tail_corrected']
y_tail_corrected=df_head_tail_center_coords['y_tail_corrected']
x_center_pos=df_head_tail_center_coords['x_center']
y_center_pos=df_head_tail_center_coords['y_center']
print(df_head_tail_center_coords.tail(1))
fig3, ax3 = plt.subplots(1,1, figsize = (figsize_x,figsize_y), dpi=600)
ax3.plot(x_head_corrected,y_head_corrected,label="head",linewidth=line_width)
ax3.plot(x_tail_corrected,y_tail_corrected,label="tail",linewidth=line_width)
ax3.plot(x_center_pos,y_center_pos,label="center",linewidth=line_width)
ax3.plot(df_head_tail_center_coords['x_center'][0],df_head_tail_center_coords['y_center'][0], 'go', markersize=5)
ax3.plot(df_head_tail_center_coords['x_center'].tail(1),df_head_tail_center_coords['y_center'].tail(1), 'ro', markersize=5)
xposition = [away_from_food['x_head_corrected'].min(), away_from_food['x_head_corrected'].max()]

rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, linewidth=1, edgecolor='r', facecolor='none')
ax3.add_patch(rect)
ax3.legend()

# doing the fft

####  loading curvature data

In [ ]:
#load data
#csv_filepath='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_good_skeleton/2020-07-01_14-41-11_chemotaxisl_worm2_spline_K.csv'
csv_filepath='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_good_skeleton/2020-07-01_17-01-27_chemotaxis_worm6_spline_K.csv'
K = np.genfromtxt(csv_filepath, delimiter=',')
#to remove nan rows in the data
K=K[~np.isnan(K).any(axis=1)]


#### average over a number of segments

In [ ]:
win=5 
Kt_avg=segment_averaging(K,win)

#### define the section of the track to do the fourier transform
(determined above by looking at the track)

In [ ]:
start_frame=away_from_food.index.min()
end_frame=away_from_food.index.max()
fps=167.0 #frames per second
Kt_avg=section_to_fourier_transform(Kt_avg,start_frame,end_frame,fps)

In [ ]:
Kt_avg=Kt_avg.T

In [ ]:
Kt_avg.shape

#### fourier transform and plot

In [ ]:

#start_frame=0
#end_frame=1000
start_freq=0 #range of requencys which should be displayed
end_freq=6 #range of requencys which should be displayed
fps=167.0 #frames per second
win=5
save=0
for idx,segment in enumerate(Kt_avg):
    x_axis,y_axis=fourier_transform(Kt_avg,fps,segment)
    fourier_plot(x_axis,y_axis,start_frame,end_frame,start_freq,end_freq,win,fps,segment,idx,save)